# Experimento 01 — Baseline pre-compra

Este cuaderno corre el pipeline completo del Entregable II utilizando **solo los predictores disponibles en el momento de la compra**: estado del cliente, categoría del producto, método y valor del pago, flete estimado, fecha estimada de entrega y atributos físicos del producto. No incluye distancia geográfica ni historial del vendedor; esas extensiones se abordan en experimentos posteriores.

El resultado de esta corrida es la **línea base** contra la que se comparan las mejoras. Las tablas se persisten en `outputs/experimentos/01_baseline_pre_compra/`.

El cuaderno está pensado para ejecutarse de arriba abajo. Cada celda hace un paso puntual, de modo que al reejecutar no se reinicializa todo el flujo. En modo rápido se trabaja con 15 000 pedidos distribuidos cronológicamente; cambie `MAX_ROWS = None` para usar todo el dataset.

## 1. Entorno y datos

Detectamos si la ejecución es local o en Colab y resolvemos la ruta raíz del repositorio para que los imports de `src` funcionen igual en ambos casos. Si los CSV de Olist no están en `data/raw/` se descargan desde la API pública de Kaggle.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Cde571/LogisTech-Modelos-Simulacion-II.git'

if IN_COLAB:
    root = Path('/content/LogisTech-Modelos-Simulacion-II')
    if not root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(root)], check=True)
    else:
        subprocess.run(['git', '-C', str(root), 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
    os.chdir(root)
else:
    root = Path.cwd()
    while root.name in ('experimentos', 'notebook', 'notebooks'):
        root = root.parent
    os.chdir(root)

print('Raíz del repositorio:', root)

In [ ]:
import urllib.request
import zipfile

In [ ]:
raw = root / 'data' / 'raw'
required = raw / 'olist_orders_dataset.csv'

if not required.exists():
    raw.mkdir(parents=True, exist_ok=True)
    archive = root / 'data' / 'olist.zip'
    urllib.request.urlretrieve(
        'https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce',
        archive,
    )
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(raw)

print('CSV disponibles en data/raw:', len(list(raw.glob('*.csv'))))

## 2. Configuración y dataset a nivel de pedido

`ExperimentConfig` encapsula las decisiones reproducibles: tamaño de muestra, número de particiones de validación cruzada temporal y repeticiones del bootstrap para los intervalos de confianza. Después cargamos las nueve tablas de Olist y las agregamos en una fila por pedido con `build_order_level_dataset`, que incluye la lógica de agregación segura (sin fugas de información posteriores a la compra) y deriva los targets `delivery_time_days` y `late_delivery`.

In [ ]:
from src.models import ExperimentConfig
from src.preprocessing import build_order_level_dataset, load_olist_tables, modeling_frame

In [ ]:
MAX_ROWS = 15_000  # Use None para la corrida completa.

config = ExperimentConfig(
    max_rows=MAX_ROWS,
    cv_splits=3,
    bootstrap_iterations=500,
)
config

In [ ]:
tables = load_olist_tables(raw)
list(tables.keys())

In [ ]:
orders = build_order_level_dataset(tables)
assert orders['order_id'].is_unique, 'Se esperaba una fila por pedido'

print(f'Pedidos agregados: {len(orders):,}')
print(f'Periodo: {orders.order_purchase_timestamp.min()} a {orders.order_purchase_timestamp.max()}')
orders.head(3)

## 3. Regresión — días hasta la entrega

El objetivo `delivery_time_days` es la diferencia en días entre la compra y la entrega efectiva. `run_task` encadena muestreo temporal, partición 60/20/20, ajuste de hiperparámetros con validación cruzada, entrenamiento final, bootstrap de intervalos y análisis de importancia. Se exploran las cinco familias requeridas (Ridge, KNN, Random Forest, MLP y LinearSVR) y se compara el pipeline completo contra representaciones PCA/UMAP sobre los dos mejores modelos.

In [ ]:
from src.experiments import run_task

In [ ]:
X_reg, y_reg, dates_reg = modeling_frame(orders, 'delivery_time_days')
print(f'Filas: {len(X_reg):,} · Columnas predictoras: {X_reg.shape[1]}')
X_reg.head(3)

In [ ]:
reg = run_task('regression', X_reg, y_reg, dates_reg, config)
list(reg.keys())

In [ ]:
reg['cv_results']

In [ ]:
reg['ranking']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=reg['ranking'], x='test_MAE', y='model', ax=ax, color='#4472C4')
ax.set(title='Regresión — MAE en prueba', xlabel='MAE [días]', ylabel='Modelo')
plt.tight_layout()
plt.show()

In [ ]:
reg['feature_importance'].head(12)

In [ ]:
reg['reductions']

## 4. Clasificación — probabilidad de entrega tardía

El objetivo binario `late_delivery` vale 1 cuando la entrega real supera la fecha prometida al cliente. La partición temporal, las cinco familias de modelos y el procedimiento de evaluación son idénticos a la tarea de regresión; solo cambian las métricas (ROC-AUC como primaria, más balanced accuracy, F1, precisión y recall). Al final se grafica la matriz de confusión del campeón para inspeccionar el tipo de error dominante.

In [ ]:
X_cls, y_cls, dates_cls = modeling_frame(orders, 'late_delivery')
print(f'Filas: {len(X_cls):,}')
print(f'Prevalencia global de retraso: {y_cls.mean():.3f}')

In [ ]:
cls = run_task('classification', X_cls, y_cls, dates_cls, config)
list(cls.keys())

In [ ]:
cls['cv_results']

In [ ]:
cls['ranking']

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.barplot(data=cls['ranking'], x='test_ROC_AUC', y='model', ax=axes[0], color='#70AD47')
axes[0].set(title='Clasificación — ROC-AUC en prueba', xlabel='ROC-AUC', ylabel='Modelo')

champion = cls['ranking'].iloc[0]['model']
champion_model = cls['models'][champion]
ConfusionMatrixDisplay.from_predictions(
    cls['parts']['y_test'],
    champion_model.predict(cls['parts']['X_test']),
    display_labels=['a tiempo', 'tarde'],
    cmap='Blues',
    ax=axes[1],
    colorbar=False,
)
axes[1].set_title(f'Matriz de confusión — {champion}')

plt.tight_layout()
plt.show()

In [ ]:
cls['feature_importance'].head(12)

In [ ]:
cls['reductions']

## 5. Persistencia de resultados

Guardamos las cuatro tablas por tarea (resultados de CV, ranking, importancia de variables, reducciones) y un `summary.json` compacto en la carpeta del experimento. Estos archivos son la evidencia reproducible; el reporte final consume los `summary.json` para citar los números principales. El análisis detallado —por qué el ROC-AUC colapsa en test y cómo influye el *shift* temporal entre particiones— se documenta en el `README.md` de la misma carpeta.

In [ ]:
import json

In [ ]:
output = root / 'outputs' / 'experimentos' / '01_baseline_pre_compra'
output.mkdir(parents=True, exist_ok=True)
output

In [ ]:
for prefix, result in [('regression', reg), ('classification', cls)]:
    for name in ['cv_results', 'ranking', 'feature_importance', 'reductions']:
        result[name].to_csv(output / f'{prefix}_{name}.csv', index=False)

print('Tablas guardadas.')

In [ ]:
def _headline(result: dict, primary_col: str) -> dict:
    top = result['ranking'].iloc[0]
    return {
        'champion': top['model'],
        'test_primary': float(top[primary_col]),
        'test_ci95': [float(top['test_ci95_low']), float(top['test_ci95_high'])],
    }

summary = {
    'sample_rows': MAX_ROWS,
    'cv_splits': config.cv_splits,
    'bootstrap_iterations': config.bootstrap_iterations,
    'regression': _headline(reg, 'test_MAE'),
    'classification': _headline(cls, 'test_ROC_AUC'),
}
(output / 'summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
summary